# Two-Tower Contrastive Training (Week 1 Milestone)

This notebook trains the `TwoTowerAnomalyModel` by aligning the Sensor Tower and Visual Tower representations using the Normalized Temperature-scaled Cross Entropy (NT-Xent) contrastive loss.

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader

# Add backend to path
sys.path.append(os.path.abspath('../backend'))

from app.models.two_tower import TwoTowerAnomalyModel
from app.training.trainer import TwoTowerContrastiveTrainer
from app.training.eval import EmbeddingVisualizer
from app.data.pair_dataset import PairDataset

## 1. Setup Mock Datasets
For this experiment, we'll create mock datasets that mimic the output of `MIMIIDataset` and `MVTecDataset`.

In [ ]:
from torch.utils.data import Dataset

class MockSensorDataset(Dataset):
    def __init__(self, num_samples=100, channels=14, seq_len=50):
        self.data = torch.randn(num_samples, channels, seq_len)
        self.labels = torch.randint(0, 2, (num_samples,))
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class MockVisualDataset(Dataset):
    def __init__(self, num_samples=100, img_size=224):
        self.data = torch.randn(num_samples, 3, img_size, img_size)
        self.labels = torch.randint(0, 2, (num_samples,))
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx], None, self.labels[idx]

print("Initializing Mock Datasets...")
sensor_train = MockSensorDataset(num_samples=200)
visual_train = MockVisualDataset(num_samples=200)

sensor_val = MockSensorDataset(num_samples=50)
visual_val = MockVisualDataset(num_samples=50)


## 2. Initialize PairDataset and DataLoader

In [ ]:
pair_train = PairDataset(sensor_train, visual_train)
pair_val = PairDataset(sensor_val, visual_val)

train_loader = DataLoader(pair_train, batch_size=32, shuffle=True)
val_loader = DataLoader(pair_val, batch_size=32, shuffle=False)

print(f"Train pairs: {len(pair_train)}")
print(f"Val pairs: {len(pair_val)}")


## 3. Initialize Model and Trainer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 14 sensor channels for CMAPSS, embedding dim 256
model = TwoTowerAnomalyModel(sensor_input_dim=14, embed_dim=256, freeze_visual=True)

trainer = TwoTowerContrastiveTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    sensor_lr=3e-4,
    visual_lr=1e-4,
    device=device,
    checkpoint_dir="../backend/checkpoints/two_tower"
)


## 4. Run Joint Training Loop

In [ ]:
trainer.train()

## 5. Evaluate and Plot t-SNE

In [ ]:
visualizer = EmbeddingVisualizer(model, val_loader, device=device)
visualizer.plot_tsne(save_path="../docs/assets/tsne_week1.png")